# 003 · Comparing Text with difflib

> **Standard Library › Text Processing** · Python 3.14

`difflib` compares sequences — strings, lists of lines, lists of words — and tells you how similar they are and what changed. It powers "did you mean…?" suggestions, fuzzy matching and `git diff`-style output, all without a third-party package.

**In this notebook you will learn:**

- how `SequenceMatcher` measures similarity with `ratio()` and describes edits with `get_opcodes()`
- how to build "did you mean?" suggestions with `get_close_matches`
- how to produce unified, context and `ndiff` text diffs
- where `HtmlDiff` fits in, and the main gotchas (case, asymmetry, auto-junk)

## .NET mental model

There is **no BCL equivalent** of `difflib`. In .NET you would typically reach for a NuGet package:

| Need | .NET | Python |
|---|---|---|
| Line-by-line text diff | `DiffPlex` (NuGet) — `InlineDiffBuilder`, `SideBySideDiffBuilder` | `difflib.unified_diff`, `ndiff`, `context_diff` |
| Similarity score | `FuzzySharp` / hand-written Levenshtein | `SequenceMatcher(...).ratio()` |
| "Did you mean?" | hand-written | `difflib.get_close_matches` |
| HTML side-by-side diff | `DiffPlex` + your own rendering | `difflib.HtmlDiff` |

`difflib` uses a variant of the Ratcliff/Obershelp "gestalt pattern matching" algorithm, **not** Levenshtein distance — the scores are similar in spirit but not identical.

## `SequenceMatcher` and `ratio()`

`SequenceMatcher(isjunk, a, b)` compares two sequences of hashable items. The first argument is an optional "junk" filter — pass `None` for most uses. `ratio()` returns a similarity in `[0.0, 1.0]`: `2 * matches / total_length`.

In [ ]:
from difflib import SequenceMatcher

def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

print(round(similarity("kitten", "sitting"), 3))
print(similarity("apple", "apple"))        # identical -> 1.0
print(similarity("apple", "xyz"))          # nothing in common -> 0.0

⚠️ **Gotcha:** the comparison is **exact** — case and whitespace matter. Normalise first if you want caseless matching.

In [ ]:
print(similarity("Apple", "apple"))
print(similarity("Apple".casefold(), "apple".casefold()))

⚠️ **Gotcha:** `ratio()` is **not always symmetric**. Swapping the arguments can give a different score, because the algorithm picks matching blocks greedily from `a`'s point of view. If you need a stable score, compare in a fixed order (or take the `max` of both directions).

In [ ]:
print(similarity("tide", "diet"), similarity("diet", "tide"))

**Tip:** `ratio()` is relatively expensive. When filtering many candidates, `quick_ratio()` and `real_quick_ratio()` return fast **upper bounds** — use them to discard obvious non-matches first (this is exactly what `get_close_matches` does internally).

## What changed? `get_opcodes()`

`get_opcodes()` returns a list of `(tag, i1, i2, j1, j2)` tuples describing how to turn `a` into `b`. Tags are `'equal'`, `'replace'`, `'delete'` and `'insert'`.

In [ ]:
a, b = "kitten", "sitting"
for tag, i1, i2, j1, j2 in SequenceMatcher(None, a, b).get_opcodes():
    print(f"{tag:8} a[{i1}:{i2}]={a[i1:i2]!r:6} b[{j1}:{j2}]={b[j1:j2]!r}")

Because it works on **any sequence of hashable items**, you can diff at the word level by passing lists of words instead of strings:

In [ ]:
before = "the quick brown fox".split()
after = "the quick red fox jumps".split()

for tag, i1, i2, j1, j2 in SequenceMatcher(None, before, after).get_opcodes():
    if tag != "equal":
        print(tag, before[i1:i2], "->", after[j1:j2])

## "Did you mean…?" with `get_close_matches`

`get_close_matches(word, possibilities, n=3, cutoff=0.6)` returns up to `n` candidates whose similarity is at least `cutoff`, best first. It's the quickest way to add suggestions to a CLI or a config validator.

```csharp
// C# — no BCL helper; you'd compute a distance per candidate and sort
var best = commands.OrderBy(c => Levenshtein(c, typed)).Take(3);
```

In [ ]:
from difflib import get_close_matches

commands = ["install", "uninstall", "list", "freeze", "show", "search"]

print(get_close_matches("instal", commands))
print(get_close_matches("serch", commands))
print(get_close_matches("xyz", commands))           # no match -> empty list

In [ ]:
def suggest(typed: str, options: list[str]) -> str:
    if typed in options:
        return f"running {typed!r}"
    hint = get_close_matches(typed, options, n=1)
    return f"unknown command {typed!r}" + (f" — did you mean {hint[0]!r}?" if hint else "")

print(suggest("freez", commands))
print(suggest("deploy", commands))

Tune `cutoff` to trade recall for precision: higher values return only close matches, lower values return looser ones.

In [ ]:
fields = ["length", "width", "height", "len"]
print(get_close_matches("lenght", fields))               # default cutoff 0.6
print(get_close_matches("lenght", fields, cutoff=0.8))   # stricter
print(get_close_matches("lenght", fields, n=4, cutoff=0.1))   # looser, and up to 4 results

## Text diffs

All diff functions work on **lists of lines**. Use `str.splitlines(keepends=True)` so each line keeps its `"\n"` and the output joins back cleanly.

### `unified_diff` — the `git diff` format

In [ ]:
from difflib import unified_diff

old = """\
host = localhost
port = 8080
debug = false
""".splitlines(keepends=True)

new = """\
host = localhost
port = 9090
debug = false
timeout = 30
""".splitlines(keepends=True)

print("".join(unified_diff(old, new, fromfile="config.old", tofile="config.new")))

`n=` controls how many context lines surround each change (default 3). The functions return **generators**, so nothing is computed until you iterate — `"".join(...)` or `sys.stdout.writelines(...)` are the usual consumers.

In [ ]:
diff = unified_diff(old, new, fromfile="old", tofile="new", n=0)
print(type(diff).__name__)
print("".join(diff))

### `context_diff`

The older "context" format (`diff -c`) shows the before and after blocks separately, marking changed lines with `!`.

In [ ]:
from difflib import context_diff

print("".join(context_diff(old, new, fromfile="config.old", tofile="config.new")))

### `ndiff` — human-readable, with intraline hints

`ndiff` lists every line prefixed with `"  "` (same), `"- "`, `"+ "`, and adds `"? "` guide lines pointing at the **characters** that changed within a line.

In [ ]:
from difflib import ndiff

print("".join(ndiff(["apple\n", "banana\n", "cherry\n"],
                    ["apple\n", "bananas\n", "cherry\n"])))

**Tip:** `difflib.restore(ndiff_output, 1)` or `restore(..., 2)` rebuilds the original or the new sequence from an `ndiff` result.

## `HtmlDiff`

`HtmlDiff().make_file(a, b)` returns a complete HTML page with a side-by-side, colour-coded table; `make_table(a, b)` returns just the `<table>` to embed in your own page. In a real script you'd write the string to a `.html` file and open it in a browser.

In [ ]:
from difflib import HtmlDiff

html = HtmlDiff(wrapcolumn=60).make_table(old, new, fromdesc="old", todesc="new")
print(html.lstrip()[:60], "...")
print(len(html), "characters of HTML")

In Jupyter you can even render it inline with `IPython.display.HTML(html)`.

## Auto-junk

⚠️ **Gotcha:** for sequences of **200 or more items**, `SequenceMatcher` automatically treats any item that appears in more than 1% of `b` as "junk" (the *autojunk* heuristic). That keeps large diffs fast, but it can produce surprising results — e.g. lines that are just `"}"` or blank lines in source code. Pass `autojunk=False` if results look wrong.

In [ ]:
a = "The quick brown fox jumps over the lazy dog. " * 6     # 270 characters
b = a.replace("quick", "slow").replace("lazy", "sleepy")

print(len(a), len(b))
print(round(SequenceMatcher(None, a, b).ratio(), 3))                   # common letters became junk!
print(round(SequenceMatcher(None, a, b, autojunk=False).ratio(), 3))   # the sensible answer

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Similarity score 0–1 | FuzzySharp / custom | `SequenceMatcher(None, a, b).ratio()` |
| Edit operations | DiffPlex `Diff.CreateDiffs` | `SequenceMatcher(...).get_opcodes()` |
| "Did you mean?" | custom | `get_close_matches(word, options)` |
| Unified (`git`) diff | DiffPlex + formatter | `unified_diff(a_lines, b_lines)` |
| Context diff | — | `context_diff(a_lines, b_lines)` |
| Human-readable diff | DiffPlex `InlineDiffBuilder` | `ndiff(a_lines, b_lines)` |
| HTML side-by-side | DiffPlex `SideBySideDiffBuilder` | `HtmlDiff().make_file(a, b)` |
| Lines with line endings | `File.ReadAllLines` (strips them) | `text.splitlines(keepends=True)` |

## Exercises

1. Write `best_match(name, choices)` that returns the single closest choice ignoring case, or `None` if nothing scores at least 0.7.
2. Given two versions of a short shopping list (as multi-line strings), print a unified diff with zero context lines.
3. Using `SequenceMatcher` on lists of words, print only the words that were **added** between `"red green blue"` and `"red yellow green blue black"`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from difflib import SequenceMatcher, get_close_matches, unified_diff

# 1 — compare casefolded keys, return the original spelling
def best_match(name: str, choices: list[str]) -> str | None:
    lookup = {c.casefold(): c for c in choices}
    hits = get_close_matches(name.casefold(), list(lookup), n=1, cutoff=0.7)
    return lookup[hits[0]] if hits else None

products = ["Keyboard", "Keypad", "Monitor"]
print(best_match("KEYBORD", products))
print(best_match("Mouse", products))

# 2
v1 = "milk\neggs\nbread\n"
v2 = "milk\nbread\nbutter\n"
print("".join(unified_diff(v1.splitlines(keepends=True), v2.splitlines(keepends=True),
                           fromfile="v1", tofile="v2", n=0)))

# 3 — 'insert' and the b-side of 'replace' are additions
a = "red green blue".split()
b = "red yellow green blue black".split()
added = [w for tag, i1, i2, j1, j2 in SequenceMatcher(None, a, b).get_opcodes()
         if tag in ("insert", "replace") for w in b[j1:j2]]
print(added)
```

</details>